# Homework 4: Evaluation

Questions: [homework.md](../../course/cohorts/2026/homework/04-evaluation/homework.md)

## Setup

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
print(documents[0]["content"][:50] + "...")
print(documents[0]["filename"])

# Introduction

Video: [Watch this lesson](https:/...
01-agentic-rag/lessons/01-intro.md


## Generating ground truth

In [7]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(documents[0])

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
# pydantic is like zod
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [12]:
response = openai_client.responses.parse(
    model="gpt-6-luna",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result.questions)

['How does RAG help an LLM answer questions using information it wasn’t trained on?', 'What does an LLM do when you give it a prompt?', 'Why might an LLM give an incorrect answer or lack information about your documents?', 'What kind of FAQ assistant will we build in this module?', 'How does the agentic part of the course change the way the system searches?']


In [14]:
from evaluation_utils import llm_structured

def generate_questions(doc):
      user_prompt = json.dumps(doc)

      result, usage = llm_structured(
          openai_client,
          data_gen_instructions,
          user_prompt,
          Questions
      )

      return result, usage

In [15]:
result, usage = generate_questions(documents[0])
usage

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}), output_tokens=169, output_tokens_details=OutputTokensDetails(reasoning_tokens=69), total_tokens=1189)

In [16]:
result, usage = generate_questions(documents[1])
usage

ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cache_write_tokens=1283, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 1283}), output_tokens=161, output_tokens_details=OutputTokensDetails(reasoning_tokens=50), total_tokens=1447)

In [17]:
result, usage = generate_questions(documents[2])
usage

ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cache_write_tokens=1750, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 1750}), output_tokens=171, output_tokens_details=OutputTokensDetails(reasoning_tokens=54), total_tokens=1924)

## The full ground truth

In [18]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)

360

In [19]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Searching the chunks

In [20]:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=lambda key: scores[key], reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [22]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)


In [23]:
def text_search(query, num_results=5):
  return index.search(query, num_results=num_results)

In [24]:
from minsearch import VectorSearch
from embedder import Embedder

vindex = VectorSearch()

texts = [chunk["content"] for chunk in chunks]

embed = Embedder()
X = embed.encode_batch(texts)

vindex.fit(X, chunks)


def vector_search(query, num_results=5):
  vector = embed.encode(query)
  return vindex.search(vector, num_results=num_results)


In [25]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [26]:
q = ground_truth[0]["question"]

In [27]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [28]:
text_search(q, 1)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

## Q3. First result with vector search

In [29]:
vector_search(q, 1)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Evaluation metrics

In [30]:
def compute_relevance(query, search_function):
    filename = query["filename"]
    results = search_function(query=query["question"])

    relevance = []
    for result in results:
        relevance.append(int(result["filename"] == filename))

    return relevance

In [31]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for entry in tqdm(ground_truth):
        relevance = compute_relevance(entry, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [33]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

## Q4. Evaluating text search

In [34]:
relevance = compute_relevance_total(ground_truth, text_search)
hit_rate(relevance)

  0%|          | 0/360 [00:00<?, ?it/s]

0.7583333333333333

## Q5. Evaluating vector search

## Q6. Tuning hybrid search